# DPO Preference Alignment for HR Assistant

This notebook performs DPO (Direct Preference Optimization) alignment
using the preference dataset to improve response quality after SFT.

In [ ]:
# Install required packages
%pip install -q unsloth transformers datasets accelerate peft trl bitsandbytes

# For Colab, you may need to restart runtime after this cell

In [ ]:
import torch
from unsloth import FastLanguageModel
from datasets import Dataset
import json

# Check GPU availability
print(f"GPU available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Load preference dataset
preference_data = []
with open('../../data/preference_dataset.jsonl', 'r') as f:
    for line in f:
        preference_data.append(json.loads(line))

print(f"Loaded {len(preference_data)} preference examples")

# Show samples
for i, item in enumerate(preference_data[:3]):
    print(f"Example {i+1}:")
    print(f"  Prompt: {item['prompt']}")
    print(f"  Chosen: {item['chosen'][:100]}...")
    print(f"  Rejected: {item['rejected']}")
    print()

In [ ]:
# Format for DPO training
def format_dpo(example):
    # Format prompt with instruction template
    prompt = f"### Instruction:\n{example['prompt']}\n\n### Response:\n"
    return {
        "prompt": prompt,
        "chosen": example["chosen"],
        "rejected": example["rejected"]
    }

dataset = Dataset.from_list(preference_data)
dataset = dataset.map(format_dpo)

# Split
dataset = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(f"Train samples: {len(train_dataset)}")
print(f"Eval samples: {len(eval_dataset)}")

In [ ]:
# Load SFT model (from instruction fine-tuning)
max_seq_length = 512
dtype = None
load_in_4bit = True

# Option 1: Load from instruction FT adapter
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="../../models/instruction_ft_adapter",
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

# Option 2: If merged model exists
# model, tokenizer = FastLanguageModel.from_pretrained(
#     model_name="../../models/instruction_ft_merged",
#     max_seq_length=max_seq_length,
#     dtype=dtype,
#     load_in_4bit=load_in_4bit,
# )

In [ ]:
# Apply LoRA for DPO (can reuse existing LoRA or add new)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
# DPO Training arguments
from trl import DPOTrainer
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="../../models/dpo_aligned",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    max_steps=100,
    learning_rate=5e-5,  # Lower LR for DPO
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=10,
    evaluation_strategy="steps",
    eval_steps=20,
    save_steps=50,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=3407,
    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
# Initialize DPO Trainer
trainer = DPOTrainer(
    model=model,
    ref_model=None,  # Use model as reference (PeftModel handles this)
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    tokenizer=tokenizer,
    max_length=512,
    max_prompt_length=256,
    beta=0.1,  # DPO temperature parameter
)

In [ ]:
# Train with DPO
trainer_stats = trainer.train()

In [ ]:
# Save the DPO-aligned model
model.save_pretrained("../../models/dpo_aligned_adapter")
tokenizer.save_pretrained("../../models/dpo_aligned_adapter")

# Save merged model for inference
model.save_pretrained_merged("../../models/dpo_aligned_merged", tokenizer, save_method="merged_16bit")

In [ ]:
# Test the DPO-aligned model
FastLanguageModel.for_inference(model)

test_questions = [
    "How can I apply for sick leave?",
    "What is the work from home policy?",
    "How does reimbursement work?",
    "What is the notice period for resignation?",
    "What employee benefits are available?",
    "How is overtime calculated?",
    "What is the onboarding process?",
    "How do I report a compliance concern?",
    "What is the attendance policy?",
    "How are performance reviews conducted?"
]

print("=== DPO-ALIGNED MODEL RESPONSES ===\n")
dpo_responses = {}
for q in test_questions:
    prompt = f"### Instruction:\n{q}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=150, temperature=0.7, do_sample=True)
    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    response = generated.split("### Response:\n")[-1]
    dpo_responses[q] = response
    print(f"Q: {q}")
    print(f"A: {response}")
    print("---")

In [ ]:
# Save all three model responses for final evaluation
import json

# Load previous comparison data
with open('../../reports/sft_comparison_data.json', 'r') as f:
    comparison_data = json.load(f)

comparison_data["dpo_aligned"] = dpo_responses

with open('../../reports/final_comparison_data.json', 'w') as f:
    json.dump(comparison_data, f, indent=2)

print("Final comparison data saved to reports/final_comparison_data.json")